# Modelo recomendación

In [1]:
import json
import numpy as np
import os
import pandas as pd
from google.cloud import storage
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from tqdm.notebook import tqdm

In [2]:
load_dotenv()
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "bubbo-dfba0-47e395cdcdc7.json"

# ========== Load environment variables ========== (parametros correctos)
PROJECT = "bubbo-dfba0"
LOCATION = "europe-southwest1"
#(donde están los embeddings por si queres consultar alguno)
BUCKET = "gs://embeddings_new_bucket"
#(donde esta el index para encontrar los knn)
INDEX_NAME = "indices_netflix2"
# (el endpoint para el index)
ENDPOINT_ID = "projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272"
# ========== Load model ========== (no sé si lo usaremos, pero es el modelo a usar si vas a generar embeddings)
MODEL_ID = "text-multilingual-embedding-002"
vertexai.init(project=PROJECT, location=LOCATION)

In [3]:
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import MatchingEngineIndexEndpoint
from google.cloud import aiplatform

#### Funciones para crear y desplegar índices en Vertex AI

El endpoint utilizado siempre es el mismo, lo que varía de cada vez son los índices. En este notebook se usa el índice "indices_netflix_2" de Vertex AI.

In [4]:
index_endpoint = MatchingEngineIndexEndpoint(
    index_endpoint_name=ENDPOINT_ID,
    project=PROJECT,
    location=LOCATION
)

In [5]:
# Conectar al índice que ya creaste (usa el ID del índice que creaste antes)
INDEX_RESOURCE_NAME = "projects/75629471929/locations/europe-southwest1/indexes/7476415186086133760"
index = aiplatform.MatchingEngineIndex(INDEX_RESOURCE_NAME)

In [6]:
# Desplegar el índice en el endpoint
deployed_index = index_endpoint.deploy_index(
    index=index,
    deployed_index_id="netflix_movies_index",
    display_name="Netflix MOvies Index",
    machine_type="e2-standard-16",      # Mantener este tipo
    min_replica_count=1,               # Solo 1 réplica mínima
    max_replica_count=2,               # Máximo 2 réplicas (en lugar de 10)
    enable_access_logging=False        # Desactivar logging para ahorrar
)

Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272
Deploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272/operations/8642246056714698752
MatchingEngineIndexEndpoint index_endpoint Deployed index. Resource name: projects/75629471929/locations/europe-southwest1/indexEndpoints/5785049643217846272


In [10]:
from vertexai.language_models import TextEmbeddingModel
from tqdm import tqdm

#### Función para extraer el tmdb_id del "ExternalIDs" de un jsonl que ha sido cargado

In [ ]:
def extract_tmdb_id(external_ids):
    """
    Extrae el tmdb_id del array ExternalIds
    """
    # Verificar que external_ids no sea None y sea una lista
    if not external_ids or not isinstance(external_ids, list):
        return None
        
    for ext_id in external_ids:
        if ext_id and ext_id.get("Provider") == "tmdb":
            return ext_id.get("ID")
    return None


#### Función para crear un maestro de embeddings a partir de un jsonl 

In [ ]:

# PASO 1: Crear el "maestro de embeddings"
def create_master_embeddings():
    """
    Genera embeddings para TODAS las películas del catálogo
    """
    # Cargar modelo
    model = TextEmbeddingModel.from_pretrained(MODEL_ID)
    
    # Procesar TODAS las películas de Netflix (y otras plataformas)
    all_movies = []
    processed_count = 0
    skipped_count = 0
    
    # Leer archivo JSONL
    with open('es_netflix.jsonl', 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f):
            try:
                movie = json.loads(line)
                synopsis = movie.get("Synopsis", "")
                external_ids = movie.get("ExternalIds")
                
                # Verificar que tenemos los datos necesarios
                if not synopsis:
                    skipped_count += 1
                    continue
                    
                tmdb_id = extract_tmdb_id(external_ids)
                
                if tmdb_id:  # Solo procesar si tenemos tmdb_id
                    all_movies.append({
                        "tmdb_id": tmdb_id,
                        "title": movie.get("Title", "Sin título"),
                        "synopsis": synopsis,
                        "genres": movie.get("Genres", []),
                        "year": movie.get("Year"),
                        "type": movie.get("Type", "")
                    })
                    processed_count += 1
                else:
                    skipped_count += 1
                    
            except json.JSONDecodeError as e:
                print(f"❌ Error JSON en línea {line_num + 1}: {e}")
                skipped_count += 1
                continue
            except Exception as e:
                print(f"❌ Error inesperado en línea {line_num + 1}: {e}")
                skipped_count += 1
                continue
    
    print(f"📊 Estadísticas:")
    print(f"   ✅ Películas procesadas: {processed_count}")
    print(f"   ⚠️ Películas omitidas: {skipped_count}")
    print(f"   📽️ Total a procesar: {len(all_movies)}")
    
    if not all_movies:
        print("❌ No se encontraron películas válidas para procesar")
        return []
    
    # Generar embeddings en lotes
    batch_size = 20
    total_batches = (len(all_movies) + batch_size - 1) // batch_size
    
    print(f"🧠 Generando embeddings en {total_batches} lotes...")
    
    for i in tqdm(range(0, len(all_movies), batch_size), desc="Procesando"):
        batch = all_movies[i:i+batch_size]
        synopses = [movie["synopsis"] for movie in batch]
        
        try:
            # Generar embeddings
            embeddings = model.get_embeddings(synopses)
            
            # Agregar embeddings a cada película
            for j, movie in enumerate(batch):
                movie["embedding"] = embeddings[j].values
                
        except Exception as e:
            print(f"❌ Error en lote {i//batch_size + 1}: {e}")
            # Continuar con el siguiente lote
            continue
    
    return [{"tmdb_id": movie["tmdb_id"], "embedding": movie["embedding"]} for movie in all_movies]

In [12]:
# Ejecutar UNA SOLA VEZ para crear el maestro
print("🎬 Iniciando creación del maestro de embeddings...")
master_embeddings = create_master_embeddings()

🎬 Iniciando creación del maestro de embeddings...
📊 Estadísticas:
   ✅ Películas procesadas: 8722
   ⚠️ Películas omitidas: 891
   📽️ Total a procesar: 8722
🧠 Generando embeddings en 437 lotes...


Procesando: 100%|██████████| 437/437 [01:34<00:00,  4.64it/s]


#### Método para poder elegir el tmdb_id que queramos para aplicarle la recomendación

Lo que hace es aislar el embedding de la pelicula con el id que nos interesa. De esta forma, podremos meter ese embedding en la consulta y cotejarlo.

In [34]:
def dict_tmdb_embedding(external_ids):
    """
    Extrae una lista de tmdb_id del maestro de embeddings
    """
    # Verificar que external_ids no sea None y sea una lista
    if not external_ids or not isinstance(external_ids, list):
        return None
    tmdb_embedding_dict = {}    
    for ext_id in external_ids:
        id = ext_id.get("tmdb_id")
        embedding = ext_id.get("embedding")
        if id and embedding:
            tmdb_embedding_dict[id] = embedding
    return tmdb_embedding_dict

In [35]:
emb = dict_tmdb_embedding(master_embeddings)

In [ ]:
movie_tmdb_id = "475557" # Ejemplo de tmdb_id, cámbialo por el que quieras consultar
movie_embedding = emb.get(movie_tmdb_id)
len(movie_embedding)

768

### Consulta KNN. 10 pelis mas parecidas

In [30]:
# Consultar los 10 vecinos más cercanos (KNN)
response = index_endpoint.find_neighbors(
    deployed_index_id="netflix_movies_index",  # ID del índice desplegado
    queries=[movie_embedding],  # Debe ser una lista de embeddings
    num_neighbors=11
)

In [31]:
# Obtener los ids de una lista [[...]] de tuplas
recommended_ids = [neighbor.id for neighbor in response[0] if neighbor.id != movie_tmdb_id]
print(f"IDs de películas recomendadas: {recommended_ids}")

IDs de películas recomendadas: ['194662', '241259', '155', '438396', '2098', '134029', '418548', '324849', '459965', '647250']


In [32]:
with open('es_netflix.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        movie = json.loads(line)
        external_ids = movie.get("ExternalIds", [])
        tmdb_id = extract_tmdb_id(external_ids)  # Extraer TMDB ID
        
        if tmdb_id in recommended_ids:  # Comparar TMDB IDs
            print(f"Película recomendada: {movie.get('Title', 'Sin título')} (TMDB ID: {tmdb_id})")
            print(f"  Netflix ID: {movie.get('Id', 'N/A')}")
            print(f"  Géneros: {movie.get('Genres', [])}")
            print("---")

Película recomendada: El caballero oscuro (TMDB ID: 155)
  Netflix ID: 70079583
  Géneros: None
---
Película recomendada: Batman: La serie animada (TMDB ID: 2098)
  Netflix ID: 70177020
  Géneros: None
---
Película recomendada: Birdman (o la inesperada virtud de la ignorancia) (TMDB ID: 194662)
  Netflix ID: 80000643
  Géneros: ['Indie']
---
Película recomendada: Batman la Lego película (TMDB ID: 324849)
  Netflix ID: 80131731
  Géneros: None
---
Película recomendada: El precio del éxito (TMDB ID: 459965)
  Netflix ID: 80198419
  Géneros: None
---
Película recomendada: Orígenes secretos (TMDB ID: 438396)
  Netflix ID: 81038599
  Géneros: ['Thriller']
---
Película recomendada: Una historia muy real (TMDB ID: 134029)
  Netflix ID: 81132961
  Géneros: None
---
Película recomendada: Mi reno de peluche (TMDB ID: 241259)
  Netflix ID: 81219887
  Géneros: None
---
Película recomendada: Everybody Happy (TMDB ID: 418548)
  Netflix ID: 81408956
  Géneros: None
---
Película recomendada: La Máquin